# VGG, read by running it

[*Very Deep Convolutional Networks for Large-Scale Image Recognition*](https://arxiv.org/abs/1409.1556) — Simonyan & Zisserman, ICLR 2015.

This notebook works through the paper in the order the questions actually came up: build the six
architectures and check them against the paper's own tables, ask what the tables leave out, and then
re-run the paper's test-time procedure on the paper's own validation set.

**What this is not.** Nothing here is trained to convergence on ImageNet. The architectures are built
from scratch and verified; the evaluation study uses torchvision's pretrained configuration D. Where a
claim could not be reproduced, that is said rather than worked around — there is a list at the bottom.

Cells marked **⏱ needs a GPU** or **⏱ needs ImageNet** read their numbers back from `results/`. Every
other cell runs here.

---
## 1. The six configurations

Table 1's whole design is that depth is the only thing that moves. One receptive field (3×3), one
stride, one padding rule, five max-pool stages, three fully-connected layers. Hold all of that constant,
vary depth, and the error difference can be attributed to depth.

Each configuration is a list read left to right: `(k, channels)` is a convolution, `"M"` a 2×2 max-pool,
`"L"` the local response normalization that only A-LRN has.

In [1]:
import torch, torch.nn as nn
from vgg import CONFIGS, WEIGHT_LAYERS, TABLE_2, make_vgg

for name, cfg in CONFIGS.items():
    convs = [x for x in cfg if x != "M" and x != "L"]
    print(f"{name:<6} {len(convs):>2} convolutions + 3 FC = {WEIGHT_LAYERS[name]:>2} weight layers")

A       8 convolutions + 3 FC = 11 weight layers
A-LRN   8 convolutions + 3 FC = 11 weight layers
B      10 convolutions + 3 FC = 13 weight layers
C      13 convolutions + 3 FC = 16 weight layers
D      13 convolutions + 3 FC = 16 weight layers
E      16 convolutions + 3 FC = 19 weight layers


Padding is `k // 2` on every convolution, which preserves the spatial size for any odd kernel — so the
3×3 layers get padding 1 and configuration C's 1×1 layers get padding 0 from the same rule, with no
special case. Only the five pools change the resolution, which is why 224 → 7 is exactly 2⁵.

In [2]:
for name in CONFIGS:
    m = make_vgg(name)
    total = sum(p.numel() for p in m.parameters())
    ok = total == TABLE_2[name]
    print(f"{name:<6} {total:>12,}   Table 2: {TABLE_2[name]:>12,}   {'exact' if ok else 'MISMATCH'}")

A       132,863,336   Table 2:  132,863,336   exact


A-LRN   132,863,336   Table 2:  132,863,336   exact


B       133,047,848   Table 2:  133,047,848   exact


C       133,638,952   Table 2:  133,638,952   exact


D       138,357,544   Table 2:  138,357,544   exact


E       143,667,240   Table 2:  143,667,240   exact


All six match. The paper prints them rounded to millions, which is why A, A-LRN and B all read 133 there.

A and A-LRN are identical because local response normalization has no parameters. That is correct, and
it is also the first hint that Table 2 is measuring cost in the wrong currency — we come back to it in §3.

---
## 2. The 3×3 claim, and the word the cost argument leaves out

§2.3 argues that three stacked 3×3 layers see the same 7×7 receptive field as one 7×7 layer while
costing 27C² weights against 49C² — the single layer is 81% more expensive.

The cost arithmetic is easy to check. The part worth checking is the other half of the claim.

In [3]:
C = 64
print(f"three 3x3 over {C} channels: 3 * 3*3*{C}*{C} = {3*3*3*C*C:,}  (27C^2)")
print(f"one   7x7 over {C} channels:     7*7*{C}*{C} = {7*7*C*C:,}  (49C^2)")
print(f"the single layer is {49/27 - 1:.0%} more expensive")

three 3x3 over 64 channels: 3 * 3*3*64*64 = 110,592  (27C^2)
one   7x7 over 64 channels:     7*7*64*64 = 200,704  (49C^2)
the single layer is 81% more expensive


**Convolution is linear.** So two stacked 3×3 convolutions with *no activation between them* are not a
deeper model at all — they collapse exactly into one 5×5 convolution. The depth argument in §2.3 is an
argument about the ReLUs, not about the kernels, and the cost comparison never mentions them.

The cleanest way to see it is the impulse response: feed a single bright pixel through the stack and
look at what comes out.

In [4]:
torch.manual_seed(0)
a = nn.Conv2d(1, 1, 3, padding=1, bias=False)
b = nn.Conv2d(1, 1, 3, padding=1, bias=False)

impulse = torch.zeros(1, 1, 9, 9); impulse[0, 0, 4, 4] = 1.0
with torch.no_grad():
    stacked = b(a(impulse))            # no activation between them

# the response is 5x5 and nonzero nowhere else: the pair IS a single 5x5 kernel
nz = (stacked.abs() > 1e-7)[0, 0]
rows, cols = nz.any(1).nonzero().flatten(), nz.any(0).nonzero().flatten()
print(f"impulse response occupies rows {rows.min().item()}-{rows.max().item()}, "
      f"cols {cols.min().item()}-{cols.max().item()}  ->  a {rows.max()-rows.min()+1} x "
      f"{cols.max()-cols.min()+1} kernel")

equivalent = nn.Conv2d(1, 1, 5, padding=2, bias=False)
with torch.no_grad():
    equivalent.weight.copy_(stacked[:, :, 2:7, 2:7].flip(-1).flip(-2))
    same = torch.allclose(equivalent(impulse), stacked, atol=1e-6)
print("one 5x5 layer reproduces the pair exactly:", same)

impulse response occupies rows 2-6, cols 2-6  ->  a 5 x 5 kernel
one 5x5 layer reproduces the pair exactly: True


Two layers in, one layer out, same function. Put a ReLU between them and no single convolution can
reproduce it — that is the entire content of "deeper".

---
## 3. What Table 2 cannot see

**⏱ needs a GPU.** Numbers below are from one NVIDIA A100-SXM4-80GB, batch 32, forward + backward,
median of 25 steps after warmup. Reproduce with `python cost.py`.

Table 2 quantifies cost as parameters. A and A-LRN have the same parameter count and the same
arithmetic, and the paper's only remark on what LRN costs is that it "leads to increased memory
consumption and computation time" (§4.1), with no figure attached.

In [5]:
import json
cost = json.load(open("results/cost.json"))
lrn = cost["lrn"]

print(f"{'':<14}{'A':>16}{'A-LRN':>16}{'change':>14}")
for label, key, fmt in [("parameters", "params", "{:,}"), ("mult-adds", "gmacs", "{} G"),
                        ("step time", "step_ms", "{} ms"), ("peak memory", "peak_mb", "{:,} MB")]:
    a, b = lrn["A"][key], lrn["A_LRN"][key]
    d = "none" if a == b else ("+%.1f%%" % lrn["delta"]["step_ms_pct"] if key == "step_ms"
                               else "+{:,} MB".format(lrn["delta"]["peak_mb"]))
    print(f"{label:<14}{fmt.format(a):>16}{fmt.format(b):>16}{d:>14}")

w = lrn["why"]
print(f"\nPyTorch expands LocalResponseNorm into {w['autograd_ops_in_pytorch_lrn']} autograd operations.")
print(f"One full-size tensor at this batch is {w['one_full_size_tensor_mb']} MB; "
      f"the increase is {w['ratio']}x that.")

                             A           A-LRN        change
parameters         132,863,336     132,863,336          none
mult-adds               7.61 G          7.61 G          none
step time             27.45 ms        46.02 ms        +67.7%
peak memory           4,624 MB        6,619 MB     +1,995 MB

PyTorch expands LocalResponseNorm into 10 autograd operations.
One full-size tensor at this batch is 392 MB; the increase is 5.09x that.


The first two rows are what Table 2 reports. The last two are what it cannot.

The authors used Caffe, which had a fused LRN kernel — so this cost is this framework's, not theirs,
and there is no honest way to project it backwards onto their machines. What it does establish is that
a layer with no parameters and no arithmetic is not free, and a table of parameter counts will never say so.

### A lesson about which number you measure

`nn.ReLU(inplace=True)` should save memory: VGG-16 at batch 32 stores a lot of activations. Predicted
saving ≈ 784 MB.

In [6]:
ip = cost["inplace_relu"]
print(f"predicted saving        {ip['predicted_saving_mb']:>8} MB")
print(f"measured, global peak   {ip['measured']['saving_mb']:>8} MB")
print(f"measured, forward peak  {ip['forward_peak']['saving_mb']:>8} MB")
print("\n" + ip["reading"])

predicted saving             784 MB
measured, global peak        0.0 MB
measured, forward peak     391.7 MB

The saving is real and it is in the forward pass. It is invisible in the global peak because the global peak is set during backward, when the stored activations are being consumed anyway. Measuring only the global peak reports zero and hides a 392 MB effect.


Both measurements are right. The saving is real and it is in the forward pass; it is invisible in the
global peak because that peak is set during *backward*, when the stored activations are being consumed
anyway. Report only the global peak and you report zero for an effect of several hundred megabytes.

### And a second effect: a multiply-accumulate is not a multiply-accumulate

In [7]:
print(f"{'step':<9}{'+GMACs':>9}{'ms/GMAC':>10}   what changed")
for s in cost["arithmetic_intensity"]["steps"]:
    print(f"{s['step']:<9}{s['gmacs']:>9.2f}{s['ms_per_gmac']:>10.2f}   {s['changed']}")

step        +GMACs   ms/GMAC   what changed
A to B        3.70      4.12   second conv in blocks 1–2 (224², 112²)
B to C        0.46      4.77   three 1 × 1 convs (56, 28, 14)
C to D        3.70      1.66   the same three, now 3 × 3
D to E        4.16      2.04   fourth conv in blocks 3–5


C→D adds **exactly the same** 3.70 GMACs as A→B and costs 2.5× less per GMAC, because it is added late
on small maps rather than early on large ones. And C's 1×1 convolutions are the worst value in the
table: a 3×3 kernel does nine multiply-accumulates for every weight it loads, a 1×1 does one.

GMAC count alone does not predict time.

![](figures/cost.svg)

---
## 4. §3.1 — why the deep configurations needed pre-initialisation

**⏱ needs a GPU and Imagenette.** Reproduce with `python init_study.py --data path/to/imagenette2`.

> "The initialisation of the network weights is important, since bad initialisation can stall learning
> due to the instability of gradient in deep nets. To circumvent this problem, we began with training
> the configuration A ... Then, when training deeper architectures, we initialised the first four
> convolutional layers and the last three fully-connected layers with the layers of net A."

Read once, that is an engineering workaround. Run it and it is the difference between a network that
learns and one that does not.

In [8]:
h = json.load(open("results/init_study.json"))["heatmap"]
print(f"Configuration E, {h['epochs']} epochs on Imagenette. Chance is {h['chance_pct']}%.\n")
print(f"{'initialisation':<22}" + "".join(f"{lr:>9.0e}" for lr in h["learning_rates"]))
for row in h["rows"]:
    cells = "".join("      NaN" if n else f"{a:9.1f}" for a, n in zip(row["acc"], row["nan"]))
    print(f"{row['init']:<22}{cells}")
s = h["summary"]
print(f"\n{s['cells_that_train']} cells train, {s['cells_near_chance']} sit at chance, "
      f"{s['cells_diverged']} diverge.")

Configuration E, 5 epochs on Imagenette. Chance is 10.0%.

initialisation            1e-03    3e-03    1e-02    3e-02    1e-01
paper init (std 0.1)        NaN      NaN      NaN      NaN      NaN
paper init (std 0.01)       9.1      9.1      9.1      9.9      9.9
Glorot                      9.1      9.1      9.9      9.9      9.9
Kaiming                    50.1     48.7      9.9      NaN      NaN

2 cells train, 11 sit at chance, 7 diverge.


Two cells out of twenty. Only Kaiming, and only at the two lowest rates — and its window has two
*different* edges: at 1e-2 the network settles at chance, from 3e-2 upward it blows up. Depth narrows
the window from both sides at once.

Glorot gives half Kaiming's variance per layer, so the shortfall in standard deviation compounds as a
power of the depth.

In [9]:
sd = json.load(open("results/init_study.json"))["signal_deficit"]
for b in sd["bars"]:
    print(f"{b['case']:<16}{b['init']:<12}{b['layers']:>3} layers   "
          f"{b['deficit']:>9,}x   measured {b['measured_val_acc']:>5}%")
print(f"\nthreshold between training and not: between {sd['threshold_band'][0]}x "
      f"and {sd['threshold_band'][1]}x")
ck = sd["checked_against_measurement"]
print(f"\n{ck['reading']}")

A alone         Glorot        8 layers          16x   measured  48.9%
E + pre-init    Glorot       12 layers          64x   measured  49.9%
E alone         Glorot       16 layers         256x   measured   9.1%
E + pre-init    paper init   12 layers       6,685x   measured   9.9%
E alone         paper init   16 layers     125,933x   measured   9.1%

threshold between training and not: between 64x and 256x

One of the five bars has been compared with a measurement and the model understates it six-fold. Read the axis as an ordering, not a physical quantity. The threshold band is set by the measured outcomes, not by the model.


Survivable over 8 layers, not over 16. The paper's transplant removes four bad layers, and four is enough.

**One part could not be reproduced.** The paper trains its donor A with its own random initialisation —
and that initialisation does not train configuration A here either, at any rate tried. So the paper's
donor does not exist to copy from. The donor used is Kaiming-trained, which makes this a test of whether
a *trained A* rescues E, not a test of the paper's method end to end.

The deficit figures are **modelled**, and one of them has been checked against a measurement: the model
understates it six-fold. Read that axis as an ordering, not a quantity.

![](figures/init.svg)

---
## 5. §3.2 — the one layer that ever needed 224

Every description of VGG begins by saying it takes a 224×224 input. Only one layer in it ever required
that. The convolutions never cared — they slide, and a bigger input just gives a bigger output. The
constraint is the first fully-connected layer, which demands exactly 7·7·512 = 25,088 numbers.

§3.2 removes that one constraint:

> "the fully-connected layers are first converted to convolutional layers (the first FC layer to a
> 7 × 7 conv. layer, the last two FC layers to 1 × 1 conv. layers)"

In [10]:
fc = nn.Linear(25088, 4096)
cv = nn.Conv2d(512, 4096, kernel_size=7)
print(f"Linear weight {tuple(fc.weight.shape)}  ->  {fc.weight.numel():,} values")
print(f"Conv2d weight {tuple(cv.weight.shape)}  ->  {cv.weight.numel():,} values")
print(f"\n512 * 7 * 7 = {512*7*7:,}   the same numbers, folded differently")

Linear weight (4096, 25088)  ->  102,760,448 values
Conv2d weight (4096, 512, 7, 7)  ->  102,760,448 values

512 * 7 * 7 = 25,088   the same numbers, folded differently


The reshape is legal only because the flatten feeding the `Linear` layer runs in C order over a
(512, 7, 7) tensor, so column `c*49 + h*7 + w` of the matrix is element `(c, h, w)` of the kernel. Under
any other flatten order `.view()` would still succeed, raise nothing, and place every weight somewhere
wrong — the network would run and return confident nonsense.

So the conversion is asserted, not assumed:

In [11]:
from vgg import to_dense, score_map_size

torch.manual_seed(0)
model = make_vgg("D").eval()
dense = to_dense(model)

x = torch.randn(2, 3, 224, 224)
with torch.no_grad():
    gap = (model(x) - dense(x).flatten(1)).abs().max().item()
print(f"max disagreement at 224: {gap:.3e}   (float32 round-off, not an approximation)")
print("parameter count preserved:",
      sum(p.numel() for p in model.parameters()) == sum(p.numel() for p in dense.parameters()))

max disagreement at 224: 1.211e-08   (float32 round-off, not an approximation)
parameter count preserved: True


Exactly zero would be the suspicious answer: a matrix multiply and a convolution accumulate in different
orders.

Now the payoff. Once the classifier is a convolution it can slide, and a larger input produces a **class
score map** instead of a single prediction.

In [12]:
print(f"{'input':>12}{'score map':>14}{'predicted':>12}{'positions':>11}")
for h_, w_ in [(224, 224), (256, 256), (384, 384), (512, 512), (256, 341)]:
    with torch.no_grad():
        got = tuple(dense(torch.randn(1, 3, h_, w_)).shape[2:])
    want = score_map_size(h_, w_)
    assert got == want
    print(f"{h_:>6} x {w_:<4}{got[0]:>6} x {got[1]:<6}{want[0]:>5} x {want[1]:<6}{got[0]*got[1]:>10}")

       input     score map   predicted  positions


   224 x 224      1 x 1         1 x 1              1


   256 x 256      2 x 2         2 x 2              4


   384 x 384      6 x 6         6 x 6             36


   512 x 512     10 x 10       10 x 10           100


   256 x 341      2 x 4         2 x 4              8


At 224 the map is a single position — dense evaluation there *is* the original network. The last row is
the realistic case: the paper rescales the smaller side to Q and leaves the aspect ratio alone, so real
images give rectangular maps.

**The network's stride is 32**, so each score-map position is a 224×224 window offset 32 pixels from the
last. Dense evaluation is multi-crop at every 32-pixel offset — for the cost of one pass, because the
convolutional work underneath every window is shared instead of recomputed.

![](figures/conversion.svg)

---
## 6. Table 5, on the paper's own validation set

**⏱ needs ImageNet.** All 50,000 ILSVRC-2012 validation images, torchvision's pretrained VGG-16,
float32. Reproduce with `python dense_eval.py --data path/to/imagenet/val`.

Before any comparison, the pipeline was checked against something external — torchvision publishes a
top-1 accuracy for these exact weights under exactly this protocol.

In [13]:
d = json.load(open("results/dense_eval.json"))
pc = d["pipeline_check"]
print(f"measured      {pc['measured_top1_error_pct']}% top-1 error")
print(f"torchvision   {pc['torchvision_reports_pct']}%")
print(f"gap           {pc['gap_pct']}%  ({pc['gap_images']} images in 50,000)")

measured      28.418% top-1 error
torchvision   28.408%
gap           0.01%  (5 images in 50,000)


Every other check in this study is internal — comparing our numbers to our other numbers. This one is not.

Now the three protocols, posteriors averaged over Q = {256, 384, 512} as §4.2 specifies:

In [14]:
ms = d["multi_scale"]
print(f"{'protocol':<20}{'top-1':>9}{'top-5':>9}{'paper':>16}{'offset':>9}")
for k in ["dense", "multicrop", "multicrop & dense"]:
    o, p = ms["measured"][k], ms["paper_table_5_config_D"][k]
    print(f"{k:<20}{o['top1']:>8.3f}%{o['top5']:>8.3f}%"
          f"{p['top1']:>9} / {p['top5']:<4}{ms['offset_from_paper_top1'][k]:>+8.3f}")

g = ms["gaps"]["dense_to_combination"]
print(f"\ndense to combination: {g['measured']} here, {g['paper']} in the paper")

protocol                top-1    top-5           paper   offset
dense                 25.242%   7.764%     24.8 / 7.5   +0.442
multicrop             25.100%   7.676%     24.6 / 7.5   +0.500
multicrop & dense     24.812%   7.420%     24.4 / 7.2   +0.412

dense to combination: 0.43 here, 0.4 in the paper


The ordering reproduces and so do the gaps. The ~0.45 point offset is the **weights**, not the protocol —
torchvision trained these with `RandomResizedCrop` rather than the paper's scale jittering, so there is
no training scale S at all. The evidence that it is the weights is that the offset barely moves across
three protocols: 0.442, 0.500, 0.412. Protocol and weight quality add rather than interact.

In [15]:
fam = d["mcnemar"]["families"][0]
print(f"{fam['name']} — {fam['tests']} tests declared in advance, "
      f"Bonferroni threshold {fam['bonferroni_alpha']}\n")
for r in fam["results"]:
    verdict = "significant" if r["p"] < fam["bonferroni_alpha"] else ""
    print(f"  {r['metric']}  {r['a']:<18} vs {r['b']:<18} "
          f"{r['only_a']:>5} / {r['only_b']:<5} p = {r['p']:.4f}  {verdict}")
print(f"\n{fam['verdict']}")

Table 5, multi-scale — 6 tests declared in advance, Bonferroni threshold 0.0083

  top-1  dense              vs multicrop           1117 / 1188  p = 0.1448  
  top-5  dense              vs multicrop            666 / 710   p = 0.2464  
  top-1  dense              vs multicrop & dense    490 / 705   p = 0.0000  significant
  top-5  dense              vs multicrop & dense    320 / 492   p = 0.0000  significant
  top-1  multicrop          vs multicrop & dense    511 / 655   p = 0.0000  significant
  top-5  multicrop          vs multicrop & dense    256 / 384   p = 0.0000  significant

The paper's headline claim holds on both metrics: the combination beats each alone. That dense and multi-crop cannot be told apart is consistent with the paper's own figures, which give the two an identical 7.5 on top-5.


![](figures/table5.svg)

---
## 7. Why every comparison above is a paired test

This is the part that nearly went wrong.

At a single scale, dense evaluation and centre-cropping score 14.73% and 14.88% top-1 on the same 3,925
images. A difference of 0.15 points — six images. Easy to explain. The explanation would have been fiction.

In [16]:
w = d["where_dense_acts"]
m = w["matrix_rows_crop_cols_dense"]
names = w["buckets"]

print("rank of the correct class — rows centre-crop, columns dense\n")
print(" " * 12 + "".join(f"{n:>11}" for n in names))
for i, row in enumerate(m):
    print(f"{names[i]:<12}" + "".join(f"{v:>11,}" for v in row))

print(f"\ntop-1  {w['top1']['crop']}% vs {w['top1']['dense']}%   "
      f"McNemar {w['top1']['mcnemar'][0]} / {w['top1']['mcnemar'][1]}, "
      f"p = {w['top1']['mcnemar'][2]}")
print(f"top-5  {w['top5']['crop']}% vs {w['top5']['dense']}%   "
      f"McNemar {w['top5']['mcnemar'][0]} / {w['top5']['mcnemar'][1]}, "
      f"p = {w['top5']['mcnemar'][2]}  significant")

rank of the correct class — rows centre-crop, columns dense

                 rank 1   rank 2-5    rank 6+
rank 1            3,246        101          0
rank 2-5             85        365         15
rank 6+              10         28         75

top-1  14.73% vs 14.88%   McNemar 101 / 95, p = 0.7211
top-5  2.88% vs 2.29%   McNemar 15 / 38, p = 0.0022  significant


The two protocols **disagree on 196 images**. Dense gets 95 right that cropping misses; cropping gets 101
right that dense misses. They cancel, and the aggregate difference hides both facts — that neither is
better, and that they are very different predictors.

Which is precisely why the paper can call them *complementary* and why combining them beats each alone.
The disagreement is the raw material of the combination.

Two more things fall out of that matrix:

- Of the 38 images dense pulls into the top five, **28 land at ranks 2–5**, not rank 1. Dense improves
  the candidate list, not the top slot — and top-1 scores rank 2 and rank 900 identically, so it reports
  nothing.
- Of the 3,347 images the centre crop already ranked first, **none** fell below rank five. Averaging over
  windows dilutes a confident right answer; it does not destroy one. (A plainer reading competes: those
  are the easy images, and easy images survive any window. The two were not separated.)

### And the reason this was run on 50,000 images and not 3,925

In [17]:
ss = d["sample_size"]
print(f"{'comparison':<34}{'3,925 images':>22}{'50,000 images':>24}")
for c in ss["comparisons"]:
    s_, l_ = c["small"], c["large"]
    print(f"{c['test'].replace(', top-1',''):<34}"
          f"{s_[0]:>8} / {s_[1]:<4} p={s_[2]:.2f}"
          f"{l_[0]:>11} / {l_[1]:<5} p={l_[2]:.4f}")
pw = ss["power"]
print(f"\n{pw['discordant_pairs_available']} discordant pairs available, "
      f"about {pw['discordant_pairs_needed']} needed -> ~{pw['images_needed']:,} images")
print(pw["reading"])

comparison                                  3,925 images           50,000 images
centre-crop vs dense                    72 / 87   p=0.27       1173 / 1823  p=0.0000
dense vs combination                    30 / 34   p=0.71        413 / 612   p=0.0000
multi-crop vs combination               27 / 36   p=0.31        462 / 546   p=0.0089

64 discordant pairs available, about 130 needed -> ~7,975 images
The smaller set was not misleading, it was silent. The experiment was about two times too small for this one claim, and the two sets look identical if only one of them is ever run.


Identical code, identical weights, identical protocols. Only the sample changes. The smaller set was not
misleading — it was **silent**, and the two look the same if only one of them is ever run.

---
## 8. Which lever actually moves the number

In [18]:
print(f"{'change':<44}{'points':>9}{'p':>10}")
for lv in d["two_levers"]["levers"]:
    name = lv["change"] + ("" if lv["on"] == "-" else f",  {lv['on']}")
    flag = "" if lv["significant"] else "   not significant"
    print(f"{name:<44}{lv['points']:>9.3f}{lv['p']:>10.4f}{flag}")

change                                         points         p
Q = 256 to {256, 384, 512},  combination        1.132    0.0000
Q = 256 to {256, 384, 512},  dense              1.100    0.0000
Q = 256 to {256, 384, 512},  multi-crop         1.012    0.0000
dense to combination                            0.430    0.0000
dense to multi-crop                             0.142    0.1448   not significant


§4.3 and a table go to choosing multi-crop over dense — worth **0.142 points**, and it does not survive
its own threshold. Averaging the same model over three test scales is worth **1.01 to 1.13 points**.

Both results are in the paper. Table 4 has the scale gain and Table 5 the protocol gain, but they sit in
different tables with different baselines and nothing invites the reader to put them side by side. Doing
so reverses which one looks important.

![](figures/levers.svg)

---
## What was and was not reproduced

**Reproduced**

- Table 1 and Table 2 — all six configurations, parameter counts exact
- §3.2's FC-to-convolutional conversion, to float32 round-off
- Score-map geometry, predicted then measured
- Table 5's ordering and gaps, on the paper's own validation set
- Table 4's claim that test-time scale jittering helps
- Table 5's implicit claim that dense and multi-crop are near-equivalent alone

**Not reproduced**

- **Training.** No network here was trained to convergence on ImageNet.
- **The paper's own initialisation** — it does not train configuration A here, so the donor its
  pre-training procedure needs could not be produced.
- **Table 4's Q = {S−32, S, S+32} finding** — these weights have no training scale S.
- **§4.4 ConvNet fusion.**

**Measured here, not in the paper**

- What A-LRN actually costs in time and memory
- Milliseconds per GMAC for each step up Table 1
- That the protocol's benefit is nearly independent of weight quality
- That the scale lever is roughly three times the protocol lever
- That dense evaluation improves the candidate list rather than the top slot

Every number above comes from `results/*.json`, written by the scripts in this folder. The figures are
drawn from the same files by `figures.py`, which measures nothing — so they regenerate on a laptop, and
each one traces back to a line in a file rather than to something typed into a plotting script once.